<a href="https://colab.research.google.com/github/lucabarattini/STAT-5703/blob/main/Data_Challenge_5_STAT5703.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


In [ ]:
# 1. Load the dataset
df = pd.read_csv('income.csv')

# 2. Keep only the relevant columns
df = df[['age', 'income']]

# 3. Clean and map the target variable
# We strip whitespace just in case the CSV has formats like ' >50K'
df['income'] = df['income'].astype(str).str.strip()
df['target'] = np.where(df['income'] == '>50K', 1, 0)

# 4. Split the data EXACTLY as instructed (No random split)
# First 40,000 rows for training, the remaining 8,842 for testing
train_df = df.iloc[:40000].copy()
test_df = df.iloc[40000:48842].copy()

print(f"Training shape: {train_df.shape}")
print(f"Testing shape: {test_df.shape}")

Training shape: (40000, 3)
Testing shape: (8842, 3)


In [ ]:
# Prepare X and y for statsmodels (adding a constant/intercept is required)
X_train = sm.add_constant(train_df['age'])
y_train = train_df['target']

# Fit the Logistic GLM
model = sm.Logit(y_train, X_train).fit()

# Get the coefficient for 'age'
age_coef = model.params['age']

# Calculate the percent increase in odds: (exp(beta) - 1) * 100
odds_increase_percent = (np.exp(age_coef) - 1) * 100

print(f"\n--- Question 1 ---")
print(f"Age coefficient: {age_coef:.4f}")
print(f"Percent increase of the odds: {odds_increase_percent:.2f}")
# This last print is the exact value you need to enter for Q1.

Optimization terminated successfully.
         Current function value: 0.523015
         Iterations 6

--- Question 1 ---
Age coefficient: 0.0381
Percent increase of the odds: 3.88


In [ ]:
# Extract the p-value for the 'age' variable
p_value_age = model.pvalues['age']

# Check if it is less than 0.05
is_significant = p_value_age < 0.05

print(f"\n--- Question 2 ---")
print(f"P-value for age: {p_value_age}")
print(f"Is age a significant predictor at 0.05 level? {is_significant}")
# If True, select 'True' on your quiz.


--- Question 2 ---
P-value for age: 0.0
Is age a significant predictor at 0.05 level? True


In [ ]:
# Prepare the test dataset
X_test = sm.add_constant(test_df['age'])
y_test = test_df['target']

# Predict probabilities on the test set
pred_probs = model.predict(X_test)

# Round predicted probabilities: 1 if > 0.5, else 0
predictions = np.where(pred_probs > 0.5, 1, 0)

# Compute accuracy: (Total Correct / Total Sample Size) * 100
correct_predictions = np.sum(predictions == y_test)
accuracy = (correct_predictions / len(y_test)) * 100

print(f"\n--- Question 3 ---")
print(f"Accuracy: {accuracy:.2f}")
# Enter this value for Q3.


--- Question 3 ---
Accuracy: 74.54


In [ ]:
# True Positives: Predicted >50K (1) AND Actual is >50K (1)
true_positives = np.sum((predictions == 1) & (y_test == 1))

# False Positives: Predicted >50K (1) BUT Actual is <=50K (0)
false_positives = np.sum((predictions == 1) & (y_test == 0))

# Compute precision: TP / (TP + FP) * 100
if (true_positives + false_positives) > 0:
    precision = (true_positives / (true_positives + false_positives)) * 100
else:
    precision = 0.0 # Edge case if the model predicted 0 positives

print(f"\n--- Question 4 ---")
print(f"Precision: {precision:.2f}")
# Enter this value for Q4.


--- Question 4 ---
Precision: 19.63
